# Trajectory Planner: A Complete Guide for New Developers

## Overview
This notebook explains the **Trajectory Planner** component for autonomous vehicles. It's a ROS2 node that controls a model car to follow a path while intelligently avoiding obstacles.

**Key Idea**: The planner uses two algorithms:
1. **Pure Pursuit** - Standard path following
2. **Dynamic Window Approach (DWA)** - Smart overtaking when obstacles are detected

## Part 1: System Architecture

### What is ROS2?
ROS2 is a **robot operating system** that lets different software components communicate. Think of it like a messaging system:
- **Publishers** send data (like "vehicle position")
- **Subscribers** receive data (like the trajectory planner listening for "obstacle detected")
- **Topics** are communication channels (like `/odom`, `/path_data`)

### Trajectory Planner Data Flow
```
INPUT TOPICS                    TRAJECTORY PLANNER NODE              OUTPUT TOPICS
─────────────────               ──────────────────────               ──────────────
/odom                           ┌─────────────────────┐              /ackermann_drive
(vehicle position)  ────────→   │ Trajectory Planner  │ ────────→   (steering + speed)
                                │                     │
/path_data                      │ Pure Pursuit or     │              /visualization_
(route waypoints)   ────────→   │ DWA Algorithm       │              marker_array
                                │                     │              (RViz display)
/obstacle_detected              └─────────────────────┘
(is obstacle there?)────────→

/vehicle_state
(is car driving?)   ────────→

/peer_veh_behavior
(obstacle position) ────────→
```

## Part 2: Core Concepts

### 1. Quaternion to Yaw Conversion
**Problem**: Robots store orientation as a *quaternion* (4 numbers: x, y, z, w). We need the *yaw angle* (rotation around z-axis) for path calculations.

**Solution**: Convert using the formula below.

In [ ]:
import math

def quaternion_to_yaw(q):
    """
    Convert a quaternion to yaw angle (rotation around z-axis).
    
    Input: q - quaternion with attributes w, x, y, z
    Output: yaw angle in radians [-π, π]
    
    Why? This formula extracts the z-rotation from the quaternion.
    It's a standard conversion used in robotics.
    """
    # These calculations come from quaternion mathematics
    siny_cosp = 2 * (q.w * q.z + q.x * q.y)      # Numerator of the angle
    cosy_cosp = 1 - 2 * (q.y * q.y + q.z * q.z)  # Denominator of the angle
    
    # atan2 gives us the angle in radians
    return math.atan2(siny_cosp, cosy_cosp)

# Example: A quaternion representing a 90-degree rotation
class SimpleQuat:
    def __init__(self, w, x, y, z):
        self.w, self.x, self.y, self.z = w, x, y, z

q = SimpleQuat(w=0.707, x=0.0, y=0.0, z=0.707)  # 90 degrees around z-axis
yaw = quaternion_to_yaw(q)
print(f"Yaw angle: {yaw:.2f} radians = {math.degrees(yaw):.2f} degrees")

### 2. Coordinate Transformation (Ego-Centric View)
**Problem**: The car's sensors give positions in world coordinates, but we need to know "is that point to my left or right?"

**Solution**: Rotate world coordinates into the car's frame of reference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def world_to_ego_frame(waypoint_x, waypoint_y, ego_x, ego_y, ego_yaw):
    """
    Convert a world-frame point to ego-frame (car's perspective).
    
    Imagine the car is looking forward. This tells you:
    - x_ego: how far ahead is the point? (positive = ahead, negative = behind)
    - y_ego: how far left/right? (positive = left, negative = right)
    """
    # Vector from car to waypoint in world frame
    dx = waypoint_x - ego_x
    dy = waypoint_y - ego_y
    
    # Rotate by -yaw to transform into car's frame
    # -yaw because we're reversing the car's rotation
    x_ego = math.cos(-ego_yaw) * dx - math.sin(-ego_yaw) * dy
    y_ego = math.sin(-ego_yaw) * dx + math.cos(-ego_yaw) * dy
    
    return x_ego, y_ego

# Example visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# World frame
ax1.set_title('World Frame')
ax1.arrow(0, 0, 1, 0.5, head_width=0.1, head_length=0.1, fc='blue', ec='blue', label='Car heading')
ax1.plot(2, 1, 'r*', markersize=15, label='Waypoint')
ax1.set_xlim(-0.5, 3)
ax1.set_ylim(-0.5, 2)
ax1.set_xlabel('X (meters)')
ax1.set_ylabel('Y (meters)')
ax1.grid()
ax1.legend()

# Ego frame
ax2.set_title('Car\'s Perspective (Ego Frame)')
ax2.arrow(0, 0, 1, 0, head_width=0.1, head_length=0.1, fc='blue', ec='blue', label='Forward direction')
car_yaw = math.atan2(0.5, 1)  # Car's heading
waypoint_ego_x, waypoint_ego_y = world_to_ego_frame(2, 1, 0, 0, car_yaw)
ax2.plot(waypoint_ego_x, waypoint_ego_y, 'r*', markersize=15, label=f'Waypoint (x={waypoint_ego_x:.2f}, y={waypoint_ego_y:.2f})')
ax2.set_xlim(-0.5, 2.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_xlabel('Forward/Backward (m)')
ax2.set_ylabel('Left/Right (m)')
ax2.grid()
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Point that was (2, 1) in world frame is now ({waypoint_ego_x:.2f}, {waypoint_ego_y:.2f}) in car's frame")
print(f"Interpretation: The waypoint is {waypoint_ego_x:.2f}m ahead and {waypoint_ego_y:.2f}m to the {'left' if waypoint_ego_y > 0 else 'right'}")

## Part 3: Pure Pursuit Algorithm (Normal Driving)

### What is Pure Pursuit?
The car looks ahead at a fixed distance ("lookahead distance") and steers to pass through that point on the path. It's like a dog following a trail by always looking just ahead.

In [ ]:
def find_lookahead_point(ego_pos, path_waypoints, lookahead_distance):
    """
    Find the point on the path that is exactly 'lookahead_distance' away.
    
    Step 1: Calculate distances from car to all path points
    Step 2: Find two waypoints where lookahead_distance is between them
    Step 3: Interpolate (find exact point between those two waypoints)
    """
    # Helper function for distance
    def distance(pos1, pos2):
        return math.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
    
    lookahead_point = None
    
    # Check each pair of consecutive waypoints
    for i in range(len(path_waypoints) - 1):
        wp1 = path_waypoints[i]
        wp2 = path_waypoints[i + 1]
        
        d1 = distance(ego_pos, wp1)  # Distance to first waypoint
        d2 = distance(ego_pos, wp2)  # Distance to second waypoint
        
        # Is the lookahead distance between these two waypoints?
        if (d1 < lookahead_distance <= d2) or (d2 < lookahead_distance <= d1):
            # Yes! Interpolate: find exact point on line segment between wp1 and wp2
            # that is exactly lookahead_distance away
            if d2 != d1:
                ratio = (lookahead_distance - d1) / (d2 - d1)  # 0 to 1
                lx = wp1[0] + ratio * (wp2[0] - wp1[0])       # Linear interpolation
                ly = wp1[1] + ratio * (wp2[1] - wp1[1])
                return (lx, ly)
    
    # If no perfect match, use the closest waypoint as fallback
    return min(path_waypoints, key=lambda wp: distance(ego_pos, wp))

# Example
path = [(0, 0), (1, 0.5), (2, 1), (3, 1.5), (4, 2)]
car_pos = (0.5, 0.2)
lookahead_dist = 1.5

lookahead_pt = find_lookahead_point(car_pos, path, lookahead_dist)
print(f"Car at {car_pos}")
print(f"Lookahead distance: {lookahead_dist}m")
print(f"Target point to steer towards: {lookahead_pt}")

### Pure Pursuit Steering Calculation

Now that we have a target point, how much should we turn the steering wheel?

In [ ]:
def pure_pursuit_steering_angle(ego_pos, ego_yaw, target_x, target_y, wheelbase, max_steering_angle_deg):
    """
    Calculate the steering angle needed to reach the target point.
    
    Concept: The steering angle is proportional to the curvature needed to reach the target.
    Think of it as: "How much do I need to turn to pass through that point?"
    """
    # Vector from car to target in world frame
    dx = target_x - ego_pos[0]
    dy = target_y - ego_pos[1]
    
    # Convert to car's reference frame
    # (same coordinate transformation we learned earlier)
    x_ego = math.cos(-ego_yaw) * dx - math.sin(-ego_yaw) * dy
    y_ego = math.sin(-ego_yaw) * dx + math.cos(-ego_yaw) * dy
    
    # If target is behind the car, no steering needed
    if x_ego <= 0:
        return 0.0
    
    # Distance from car to target
    lookahead_distance = math.sqrt(x_ego**2 + y_ego**2)
    
    # Pure Pursuit formula: Calculate curvature needed
    # curvature = (2 * lateral_error) / (lookahead_distance^2)
    curvature = (2 * y_ego) / (lookahead_distance ** 2)
    
    # Convert curvature to steering angle using wheelbase
    # steering_angle = atan(wheelbase * curvature)
    steering_angle_rad = math.atan(wheelbase * curvature)
    steering_angle_deg = math.degrees(steering_angle_rad)
    
    # Limit to maximum steering angle (car can't turn infinitely sharp)
    steering_angle_deg = max(-max_steering_angle_deg, min(steering_angle_deg, max_steering_angle_deg))
    
    return steering_angle_deg

# Example: Car following a path
car_pos = (0, 0)
car_yaw = 0  # Looking forward (0 degrees)
target = (5, 0.5)  # Waypoint slightly to the right
wheelbase = 0.5  # Distance between front and rear axles (meters)
max_steering = 30  # Car can turn up to 30 degrees

steering = pure_pursuit_steering_angle(car_pos, car_yaw, target[0], target[1], wheelbase, max_steering)
print(f"Car at {car_pos}, heading {car_yaw:.1f}°")
print(f"Target at {target}")
print(f"Steering angle needed: {steering:.2f}°")
print(f"\nInterpretation: Turn the steering wheel {steering:.2f}° to the {'right' if steering < 0 else 'left'}")

### Speed Control Based on Steering Angle

Sharp turns require slower speed for safety. Straight paths allow faster speeds.

In [ ]:
def adaptive_speed(steering_angle_deg, max_steering_angle_deg, min_speed, max_speed):
    """
    Adjust speed based on steering angle.
    
    Logic:
    - Straight (0° steering) → Maximum speed
    - Sharp turn (30° steering) → Minimum speed
    - Linear interpolation between them
    """
    # Normalize steering angle to 0-1 range
    abs_steer = min(abs(steering_angle_deg), max_steering_angle_deg)
    steer_ratio = abs_steer / max_steering_angle_deg  # 0 (straight) to 1 (sharp)
    
    # Calculate speed: reduce from max_speed to min_speed as steering increases
    speed_range = max_speed - min_speed
    target_speed = max_speed - (steer_ratio * speed_range)
    
    # Ensure speed is within bounds
    target_speed = max(min_speed, min(target_speed, max_speed))
    
    return target_speed

# Test different steering angles
fig, ax = plt.subplots(figsize=(10, 6))

steering_angles = np.linspace(-30, 30, 100)
speeds = [adaptive_speed(s, 30, 0.3, 0.4) for s in steering_angles]

ax.plot(steering_angles, speeds, 'b-', linewidth=2, label='Speed profile')
ax.axvline(0, color='g', linestyle='--', alpha=0.5, label='Straight ahead')
ax.axhline(0.4, color='r', linestyle='--', alpha=0.5, label='Max speed')
ax.axhline(0.3, color='orange', linestyle='--', alpha=0.5, label='Min speed')
ax.set_xlabel('Steering Angle (degrees)')
ax.set_ylabel('Target Speed (m/s)')
ax.set_title('Adaptive Speed Control: Slow Down on Sharp Turns')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

print("Speed profile:")
for steer in [-30, -15, 0, 15, 30]:
    spd = adaptive_speed(steer, 30, 0.3, 0.4)
    print(f"  Steering {steer:3}° → Speed {spd:.2f} m/s")

## Part 4: Dynamic Window Approach (DWA) - Overtaking

### What is DWA?
DWA is an advanced algorithm that:
1. Samples different (speed, steering) combinations
2. Simulates where the car would go with each combination
3. Scores each trajectory based on:
   - Distance to target (stay close to path)
   - Collision avoidance (don't hit obstacle)
   - Speed preference (prefer forward motion)
   - Smoothness
4. Picks the best trajectory

In [ ]:
def simulate_trajectory(v, w, ego_x, ego_y, ego_yaw, dt=0.1, predict_time=1.5):
    """
    Simulate where the car will be if we apply velocity v and angular velocity w.
    
    Parameters:
    - v: linear velocity (m/s, forward/backward)
    - w: angular velocity (rad/s, rotation rate)
    - dt: time step (20 ms)
    - predict_time: how far into future to predict (1.5 seconds)
    
    Output: List of (x, y, yaw) positions along the trajectory
    """
    trajectory = []
    x, y, yaw = ego_x, ego_y, ego_yaw
    
    # Simulate step-by-step
    num_steps = int(predict_time / dt)
    for _ in range(num_steps):
        if abs(w) < 1e-3:  # Going straight (no rotation)
            # Simple forward motion
            x += v * math.cos(yaw) * dt
            y += v * math.sin(yaw) * dt
            # yaw stays same
        else:  # Turning (curved motion)
            # Bicycle model: car follows a curved path
            # Formula from kinematics: (x, y) follows a circle
            x += (v / w) * (math.sin(yaw + w * dt) - math.sin(yaw))
            y += (v / w) * (-math.cos(yaw + w * dt) + math.cos(yaw))
            yaw = yaw + w * dt
            
            # Normalize yaw to [-π, π]
            while yaw > math.pi:
                yaw -= 2 * math.pi
            while yaw < -math.pi:
                yaw += 2 * math.pi
        
        trajectory.append((x, y, yaw))
    
    return trajectory

# Example: Simulate two trajectories
v1, w1 = 0.3, 0.0  # Go straight
v2, w2 = 0.2, 0.5  # Turn left

traj1 = simulate_trajectory(v1, w1, 0, 0, 0)  # Start at origin, heading forward
traj2 = simulate_trajectory(v2, w2, 0, 0, 0)

# Plot trajectories
fig, ax = plt.subplots(figsize=(10, 8))

# Extract x, y coordinates
traj1_x = [p[0] for p in traj1]
traj1_y = [p[1] for p in traj1]
traj2_x = [p[0] for p in traj2]
traj2_y = [p[1] for p in traj2]

ax.plot(traj1_x, traj1_y, 'b-', linewidth=2, label=f'Straight: v={v1}m/s, w={w1}rad/s')
ax.plot(traj2_x, traj2_y, 'r-', linewidth=2, label=f'Turn left: v={v2}m/s, w={w2}rad/s')

# Mark start
ax.plot(0, 0, 'ko', markersize=10, label='Start')
# Mark end
ax.plot(traj1_x[-1], traj1_y[-1], 'bs', markersize=8, label='Straight end')
ax.plot(traj2_x[-1], traj2_y[-1], 'rs', markersize=8, label='Turn end')

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Simulated Vehicle Trajectories (1.5 second prediction)')
ax.grid(True, alpha=0.3)
ax.legend()
ax.axis('equal')
plt.show()

print(f"Straight motion:")
print(f"  Start: (0.00, 0.00)")
print(f"  End:   ({traj1[-1][0]:.2f}, {traj1[-1][1]:.2f})")
print(f"\nTurning motion:")
print(f"  Start: (0.00, 0.00)")
print(f"  End:   ({traj2[-1][0]:.2f}, {traj2[-1][1]:.2f})")

In [ ]:
def evaluate_trajectory(trajectory, target_x, target_y, obstacle_x, obstacle_y, max_speed):
    """
    Score a trajectory based on multiple criteria.
    Lower score = better trajectory.
    """
    if not trajectory:
        return float('inf')
    
    final_x, final_y, final_yaw = trajectory[-1]
    
    # Cost 1: How far from target?
    target_cost = math.sqrt((final_x - target_x)**2 + (final_y - target_y)**2)
    
    # Cost 2: How close to obstacle? (collision risk)
    min_obstacle_dist = min(
        math.sqrt((x - obstacle_x)**2 + (y - obstacle_y)**2)
        for x, y, _ in trajectory
    )
    if min_obstacle_dist < 0.3:  # Too close!
        collision_cost = 10.0  # Penalize heavily
    else:
        collision_cost = max(0, (0.5 - min_obstacle_dist))  # Prefer larger clearance
    
    # Cost 3: Prefer smooth, forward motion
    # Calculate actual velocity from trajectory
    dist_traveled = math.sqrt((final_x - 0)**2 + (final_y - 0)**2)
    speed_cost = abs(dist_traveled - max_speed * 0.9) / max_speed  # Prefer normal speed
    
    # Weighted sum
    total_cost = (0.3 * target_cost +   # 30% importance: reach target
                  0.4 * collision_cost + # 40% importance: avoid collision
                  0.3 * speed_cost)      # 30% importance: smooth motion
    
    return total_cost

# Example: Compare two trajectories
traj_safe = simulate_trajectory(0.2, 0.3, 0, 0, 0)  # Avoid obstacle by turning
traj_risky = simulate_trajectory(0.3, 0.0, 0, 0, 0)  # Go straight (might hit obstacle)

target_x, target_y = 1.5, 0.5  # Where we want to go
obstacle_x, obstacle_y = 0.5, 0.0  # Obstacle position

cost_safe = evaluate_trajectory(traj_safe, target_x, target_y, obstacle_x, obstacle_y, 0.4)
cost_risky = evaluate_trajectory(traj_risky, target_x, target_y, obstacle_x, obstacle_y, 0.4)

print(f"Safe trajectory (turn): cost = {cost_safe:.2f}")
print(f"Risky trajectory (straight): cost = {cost_risky:.2f}")
print(f"\nDWA picks the safe trajectory (lower cost)" if cost_safe < cost_risky else "\nDWA picks the risky trajectory (lower cost)")

## Part 5: Overtaking State Machine

When an obstacle is detected close enough, the planner enters a 4-phase state machine to safely overtake.

In [ ]:
from enum import Enum

class OvertakePhase(Enum):
    """States during overtaking maneuver"""
    NORMAL_DRIVING = 0          # Normal Pure Pursuit path following
    LANE_CHANGE_DEPARTURE = 1   # Moving sideways away from obstacle
    PASSING_PHASE = 2           # Driving alongside obstacle
    LANE_CHANGE_RETURN = 3      # Merging back to original path

def overtaking_state_machine(ego_x, ego_y, ego_yaw, peer_x, peer_y, 
                             peer_distance, current_phase, 
                             cruising_distance_completed,
                             overtake_trigger_distance=1.5,
                             overtake_cruising_length=3.0,
                             overtake_merge_safety_distance=2.0):
    """
    Update overtaking state based on current conditions.
    
    Transitions:
    1. NORMAL_DRIVING → LANE_CHANGE_DEPARTURE: when obstacle within trigger distance
    2. LANE_CHANGE_DEPARTURE → PASSING_PHASE: when vehicle starts passing (peer goes behind)
    3. PASSING_PHASE → LANE_CHANGE_RETURN: after traveling cruising_length alongside
    4. LANE_CHANGE_RETURN → NORMAL_DRIVING: when vehicle has merged back
    """
    
    # Check if peer is behind the ego vehicle (in local frame)
    dx = peer_x - ego_x
    dy = peer_y - ego_y
    
    # Rotate to ego's frame
    peer_local_x = math.cos(-ego_yaw) * dx - math.sin(-ego_yaw) * dy
    peer_is_behind = (peer_local_x < -0.5)  # Obstacle is behind if local_x < 0
    
    # State machine logic
    if (peer_is_behind and 
        cruising_distance_completed >= overtake_cruising_length and
        peer_distance >= overtake_merge_safety_distance):
        # We've passed the obstacle and are safe to merge back
        next_phase = OvertakePhase.LANE_CHANGE_RETURN
        reason = "Completed passing - merging back"
    
    elif not peer_is_behind:
        # Obstacle is still ahead/alongside - departing lane or cruising
        next_phase = OvertakePhase.LANE_CHANGE_DEPARTURE
        reason = "Obstacle still ahead - departing lane"
    
    else:
        # Obstacle is behind - we're actively passing
        next_phase = OvertakePhase.PASSING_PHASE
        reason = "Passing alongside obstacle"
    
    return next_phase, reason

# Example scenario
print("Overtaking Scenario Walkthrough:")
print("="*50)

scenarios = [
    # (ego_x, ego_y, peer_x, peer_y, distance, cruise_dist, phase_name)
    (0, 0, 1, 0.2, 1.0, 0, "NORMAL_DRIVING"),
    (0.5, 0.5, 1, 0.2, 0.7, 0, "OBSTACLE DETECTED - triggering overtake"),
    (1, 0.7, 0.8, 0.2, 0.8, 0, "MOVING SIDEWAYS"),
    (1.5, 1.0, 0.5, 0.2, 1.3, 1.5, "PASSING ALONGSIDE"),
    (2.0, 1.3, -0.5, 0.2, 2.7, 3.5, "COMPLETED PASSING - MERGING BACK"),
]

for i, (ex, ey, px, py, dist, cruise, desc) in enumerate(scenarios, 1):
    next_phase, reason = overtaking_state_machine(
        ex, ey, 0,    # ego position and yaw
        px, py,       # peer position
        dist,         # peer distance
        OvertakePhase.NORMAL_DRIVING,  # current phase
        cruise        # cruising distance
    )
    
    print(f"\nStep {i}: {desc}")
    print(f"  Ego: ({ex:.1f}, {ey:.1f}), Peer: ({px:.1f}, {py:.1f}), Distance: {dist:.1f}m")
    print(f"  → State: {next_phase.name}")
    print(f"  → Reason: {reason}")

## Part 6: Path Pruning (Optimization)

Keeping waypoints behind the vehicle wastes computation. We continuously remove old waypoints.

In [ ]:
def prune_path(ego_x, ego_y, ego_yaw, waypoints):
    """
    Remove waypoints that are behind the vehicle.
    
    Why? Fewer waypoints = faster computation, and we don't need the past path.
    
    How? Transform each waypoint to ego frame. If x_ego < 0, it's behind.
    """
    pruned = []
    
    for wp in waypoints:
        # Vector from car to waypoint
        dx = wp[0] - ego_x
        dy = wp[1] - ego_y
        
        # Transform to ego frame
        x_ego = math.cos(-ego_yaw) * dx - math.sin(-ego_yaw) * dy
        
        # Keep if in front or very close
        if x_ego > -0.5:
            pruned.append(wp)
    
    return pruned

# Example
path = [(0, 0), (0.5, 0.1), (1, 0.2), (1.5, 0.3), (2, 0.4), (2.5, 0.5), (3, 0.6)]
ego_x, ego_y, ego_yaw = 1.5, 0.15, 0

pruned_path = prune_path(ego_x, ego_y, ego_yaw, path)

print("Original path:", path)
print(f"\nVehicle at ({ego_x}, {ego_y})")
print(f"\nAfter pruning:")
print(f"  Waypoints kept: {pruned_path}")
print(f"  Waypoints removed: {len(path) - len(pruned_path)}")
print(f"  Computation saved: {(len(path) - len(pruned_path)) / len(path) * 100:.1f}%")

## Part 7: Main Control Loop (How It All Works Together)

Here's the flow of the complete trajectory planner:

In [ ]:
print("""
MAIN CONTROL LOOP (Runs 50 times per second)
=============================================

1. CHECK SAFETY CONSTRAINTS
   ├─ Is vehicle in "Driving" state?
   ├─ Is there an obstacle detected?
   └─ Do we have valid position and path data?
   
   If any fail → STOP immediately (speed = 0, steering = 0)

2. PRUNE OLD WAYPOINTS
   └─ Remove waypoints behind the vehicle for efficiency

3. CHECK GOAL
   └─ Are we at the end of the path?
   └─ If yes → Student pickup/dropoff announcement

4. DETECT OBSTACLE AND DECIDE ALGORITHM
   │
   ├─ IF peer_distance ≤ 1.5m (trigger distance):
   │   │
   │   ├─ Set is_overtaking = True
   │   ├─ Update overtaking phase (state machine)
   │   ├─ Use DWA algorithm:
   │   │  ├─ Sample 8×8 = 64 different (velocity, steering) combinations
   │   │  ├─ Simulate each to 1.5 seconds ahead
   │   │  ├─ Score each based on:
   │   │  │  ├─ Distance to target (30% weight)
   │   │  │  ├─ Collision avoidance (40% weight)
   │   │  │  └─ Speed smoothness (30% weight)
   │   │  └─ Pick best trajectory
   │   └─ Publish control command
   │
   └─ ELSE (no obstacle or far away):
       │
       ├─ Use Pure Pursuit algorithm:
       │  ├─ Find lookahead point (1.5m ahead on path)
       │  ├─ Calculate steering angle
       │  ├─ Adjust speed based on steering (less steering = faster)
       │  └─ Smooth steering transitions (AC10)
       └─ Publish control command

5. PUBLISH VISUALIZATION
   └─ Send ego and peer vehicle positions to RViz for display

6. WAIT 20ms (for 50 Hz update rate)
   └─ Repeat from step 1
""")

## Part 8: Common Issues and Debugging

### Issue 1: Vehicle Oscillates (Jerky Steering)
**Cause**: Steering changes too suddenly
**Solution**: Steering smoothing factor (AC10)

```python
# Without smoothing: steering jumps 0° → 20°
steering = 20.0

# With smoothing factor = 0.2:
smoothed_steering = 0.8 * 0 + 0.2 * 20 = 4°  # Change gradually
smoothed_steering = 0.8 * 4 + 0.2 * 20 = 7.2°
smoothed_steering = 0.8 * 7.2 + 0.2 * 20 = 9.76°
# ... gradually reaches 20°
```

In [ ]:
def steering_smoothing_comparison():
    """Compare jerky vs smooth steering transitions"""
    
    # Target steering angle changes abruptly
    target_steering_sequence = [0, 0, 0, 20, 20, 20, -15, -15, 0, 0, 0]
    
    # Without smoothing: jerky
    jerky = target_steering_sequence.copy()
    
    # With smoothing: smooth
    smoothing_factor = 0.2
    smooth = []
    previous_steering = 0
    
    for target in target_steering_sequence:
        new_steering = (1 - smoothing_factor) * previous_steering + smoothing_factor * target
        smooth.append(new_steering)
        previous_steering = new_steering
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    
    steps = range(len(target_steering_sequence))
    ax.plot(steps, jerky, 'r-o', linewidth=2, markersize=6, label='Without Smoothing (Jerky)')
    ax.plot(steps, smooth, 'g-s', linewidth=2, markersize=6, label='With Smoothing (AC10)')
    ax.axhline(0, color='k', linestyle='-', alpha=0.2)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Steering Angle (degrees)')
    ax.set_title('Steering Smoothing: Reduces Jerky Vehicle Behavior')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    plt.show()
    
    print("Comparison:")
    print("\nWithout smoothing (jerky):")
    for i, angle in enumerate(jerky):
        print(f"  Step {i}: {angle:6.1f}° change: {abs(angle - jerky[i-1]) if i > 0 else 0:6.1f}°")
    
    print("\nWith smoothing (smooth):")
    for i, angle in enumerate(smooth):
        print(f"  Step {i}: {angle:6.1f}° change: {abs(angle - smooth[i-1]) if i > 0 else 0:6.1f}°")

steering_smoothing_comparison()

### Issue 2: Fails to Overtake / Sticks Behind Obstacle
**Cause**: Trigger distance set too high or lateral offset too small
**Solution**: Adjust parameters

In [ ]:
print("""
DEBUGGING OVERTAKING FAILURES
============================

Parameter             Default   Too High?              Too Low?              Symptom
─────────────────────────────────────────────────────────────────────────────────────

overtake_trigger_     1.5m      Starts overtaking      Starts only when      Follows
distance                         too early, wastes      very close to obstacle obstacle
                                 time

overtake_target_      0.35m     Takes very wide path,  Takes narrow path,    Can't get
lateral_offset                   might overshoot        might hit obstacle    clear

overtake_cruising_    3.0m      Takes forever to pass  Merges back too early, Unsafe
length                           might merge too close  might hit obstacle    merge
                                 while still alongside

max_speed             0.4 m/s   Too fast for turns     Too slow overall       Jerky or
                                                                              too slow

FIX: Start conservative (safe) then gradually tune:
  1. Test in simulation
  2. Increase overtake_trigger_distance by 0.1m and retestime
  3. Monitor if overtaking is smooth and safe
  4. Repeat until satisfied
""")

## Part 9: Key Acceptance Criteria (AC) Summary

These are the requirements the system must satisfy:

In [ ]:
acceptance_criteria = {
    "AC1": "Detect obstacle within 1.5m → Trigger overtaking",
    "AC2": "On detection → Set is_overtaking=True, LANE_CHANGE_DEPARTURE phase",
    "AC3": "Steer 0.35m lateral offset, maintain 3.0m cruise distance",
    "AC4": "After cruise → Transition to LANE_CHANGE_RETURN, merge safely",
    "AC5": "After merge → Set is_overtaking=False, NORMAL_DRIVING phase",
    "AC6": "Publish AckermannDrive commands (with logging)",
    "AC7": "Use Pure Pursuit for normal, DWA exclusively for overtaking",
    "AC8": "Stop immediately if obstacle detected or vehicle_state ≠ 'Driving'",
    "AC9": "Log warning if /odom or /path_data missing >1s, publish halt",
    "AC10": "Smooth steering transitions (no jerky changes)",
}

print("ACCEPTANCE CRITERIA (System Requirements)")
print("="*80)
for ac, description in acceptance_criteria.items():
    print(f"\n{ac}: {description}")

print("\n" + "="*80)
print(f"Total requirements: {len(acceptance_criteria)}")
print("All implemented and tested in unit/integration tests.")

## Summary: What We Learned

### The Big Picture
The Trajectory Planner is an **autonomous vehicle controller** that:
- Follows a pre-defined path using **Pure Pursuit** algorithm
- Detects obstacles and smoothly **overtakes** them using **DWA**
- Manages **4 phases** of overtaking for safety
- Publishes **steering and speed commands** to control the vehicle
- Runs at **50 Hz** (50 times per second) for real-time response

### Key Algorithms
1. **Quaternion to Yaw**: Convert 3D orientation to 1D angle
2. **Ego-Frame Transform**: "From world coordinates to car's perspective"
3. **Pure Pursuit**: "Look ahead and steer to that point"
4. **Adaptive Speed**: "Slow down for sharp turns"
5. **DWA**: "Try many options, pick the safest"
6. **State Machine**: "Track progress through overtaking phases"
7. **Steering Smoothing**: "Prevent jerky movements"

### For Further Learning
- Study the **ROS2 documentation** for publisher/subscriber patterns
- Research **control systems** (PID, trajectory tracking)
- Read papers on **motion planning** (RRT, Dijkstra)
- Practice with **simulation** (Gazebo, CARLA) before real vehicles

### Common Mistakes to Avoid
❌ Not checking safety constraints first
❌ Tuning parameters without simulation
❌ Jerky steering (forgetting smoothing)
❌ Ignoring vehicle dynamics (wheelbase, max angle)
❌ Publishing every tiny change (sample less frequently)

---

**Next Steps**: Run this planner in a ROS2 workspace with simulated sensors. Test each algorithm independently before combining them.